# Convert YOLOv9-s slim (ReLU6) to an EdgeTPU model for Google Coral and Frigate

This notebook turns a trained YOLOv9-s checkpoint with the **slim head** (`distlite_cls64`) into a model that runs on a Google
Coral EdgeTPU in Frigate. It is the conversion path used for this release's model
(`yolov9-s-relu6-slim_320_int8_edgetpu.tflite`).

**Pipeline:** `best.pt` → ONNX → TensorFlow SavedModel (onnx2tf) → full-int8 TFLite → `edgetpu_compiler`.

**What the model outputs (2-tensor layout, "2t", preferred for Frigate):**

| tensor | shape | content |
|---|---|---|
| boxes | `(1, 2100, 4)` int8 | ltrb distances from the anchor, in stride units (decode with `max(d, 0)`) |
| classes | `(1, 2100, 17)` int8 | class logits (sigmoid on the host, only for kept boxes) |

2100 = 40×40 + 20×20 + 10×10 anchors for a 320×320 input (strides 8, 16, 32). Compared with the older DFL head
(64 box channels decoded with a softmax over 16 bins per side), the distlite head regresses the 4 distances directly.
That removes most of the head's compute: about **6.25 ms** per frame end-to-end on a Coral, against about 9 ms for the
DFL models, at about the same accuracy. See `benchmark/RESULTS.md`.

**Requires the patched detector plugin** from
[dbro/frigate-detector-edgetpu-yolo9](https://github.com/dbro/frigate-detector-edgetpu-yolo9) with distlite support.
It picks the box decode from the box tensor's channel count (64 → DFL, 4 → distlite).

### Inputs you need
1. **A trained checkpoint**: `best.pt` from `Train_YOLOv9s_slim_EdgeTPU.ipynb` (ReLU6, `distlite_cls64` head,
   17 classes). Upload it to `/content/` or point `WEIGHTS` at a URL.
2. **Calibration images** for int8 quantization: about 256 frames that look like what your cameras send to the
   detector (day and night, some grayscale/IR, some empty scenes). Either a folder of images (`CALIB_IMAGE_DIR`) or a
   ready `.npy` array `[N, 320, 320, 3]` float32 in 0..255 RGB (`CALIB_NPY`). If you have neither, the notebook
   falls back to COCO128, which works but is not tuned for camera footage.

### Layouts
- **`2t`** (default): one box tensor + one class tensor, exactly what the Frigate plugin expects.
- **`6t`** (research): one box and one class tensor per scale, NHWC `(1,H,W,4)` / `(1,H,W,17)`. Same accuracy and
  speed (within 0.1 ms) but per-scale int8 scales; the Frigate plugin does not read it.

### ReLU6 vs hard-swish
This notebook exports **ReLU6** models, which map fully to the EdgeTPU. A hard-swish ("hs") variant is more accurate at
night, but it only runs fast on the Coral through a patched NLU lookup table, costs about **+2.9 ms** per frame, and its
export tooling is not part of this notebook.


## Step 1: Install packages


In [ ]:
# [colab-only]
# Versions this notebook was tested with (CPU is enough; no GPU needed)
!pip install -q "torch==2.8.0" "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cpu
!pip install -q "tensorflow==2.19.0" "tf_keras==2.19.0" "onnx==1.19.1" "onnxslim==0.1.98" "onnxruntime==1.23.2"
!pip install -q "onnx2tf==1.28.5" "sng4onnx==2.0.1" "onnx_graphsurgeon==0.6.1" "ai-edge-litert" "protobuf>=5"
!pip install -q pyyaml opencv-python-headless pillow seaborn pandas tqdm
print("✓ packages installed (restart the runtime if Colab asks you to)")


## Step 2: Options


In [ ]:
import os, sys, glob, json, subprocess
from pathlib import Path
import numpy as np

WORK = '/content'                    # working folder
WEIGHTS = '/content/best.pt'         # trained slim distlite checkpoint (path, or an https:// URL)
LAYOUT = '2t'                        # '2t' (Frigate) | '6t' (research)
IMG = 320                            # input size; 320 is what fits the Coral comfortably
LOGIT_CLAMP = 0.0                    # 0 = no clamp (as released). >0 clamps class logits to [-c, c] before int8
CALIB_NPY = ''                       # optional: ready calibration array [N, IMG, IMG, 3] float32 0..255 RGB
CALIB_IMAGE_DIR = ''                 # optional: folder of calibration images (jpg/png), searched recursively
CALIB_N = 256                        # number of calibration frames
CALIB_GRAY_SHARE = 0.25              # share of calibration frames converted to grayscale (night/IR coverage)
OUT_NAME = f'yolov9-s-relu6-slim_{IMG}_int8'   # output files: <OUT_NAME>.tflite, <OUT_NAME>_edgetpu.tflite
LABELS = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat',
          'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear']   # training order (labels-coco17.txt)

YOLOV9_COMMIT = '5b1ea9a'            # WongKinYiu/yolov9 commit the recipe was tested with
OUT = Path(WORK) / 'export'; OUT.mkdir(parents=True, exist_ok=True)
assert LAYOUT in ('2t', '6t')
os.chdir(WORK)


## Step 3: YOLOv9 code and the slim head definition

The checkpoint stores the model as pickled Python objects, so the YOLOv9 code and `coral_slim.py` (which defines `DualDDetectSlim`) must be importable to load it. `coral_slim.py` is the same file the training notebook writes.


In [ ]:
REPO = f'{WORK}/yolov9'
if not os.path.isdir(f'{REPO}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/WongKinYiu/yolov9.git', REPO], check=True)
subprocess.run(['git', '-C', REPO, 'checkout', '-q', YOLOV9_COMMIT], check=True)
os.makedirs(f'{WORK}/recipe', exist_ok=True)
for p in (REPO, f'{WORK}/recipe'):
    if p not in sys.path: sys.path.insert(0, p)
print('yolov9 at', subprocess.run(['git', '-C', REPO, 'rev-parse', '--short=7', 'HEAD'], capture_output=True, text=True).stdout.strip())


In [ ]:
%%writefile /content/recipe/coral_slim.py
"""Train-time slim DualDDetect heads for YOLOv9-s on the Coral EdgeTPU, plus the matching loss.
The yolov9 repo stays unmodified.

install(arch) rebinds models.yolo.DualDDetect -> DualDDetectSlim (a subclass, so parse_model's lookup, the isinstance
checks in Model.__init__ and the yaml name all resolve to it) and utils.loss_tal_dual.ComputeLoss -> ComputeLossSlim.
It must run BEFORE the model is built and BEFORE train_dual.py runs (train_dual does
`from utils.loss_tal_dual import ComputeLoss` at import time). Checkpoints pickle the head as
coral_slim.DualDDetectSlim, so this file must be importable to load them later (val, export).

Only the MAIN branch (cv4 box / cv5 cls, the one exported to the EdgeTPU) is slimmed. The aux branch (cv2/cv3,
training-only deep supervision, dropped at export) keeps the stock towers and the stock DFL loss.
  cls64          : cv5 = Conv(c,64,3) Conv(64,64,3) 1x1->nc;  cv4 = stock (Conv 64, Conv 64 g4, 1x1 g4 -> 4x16 DFL)
  distlite_cls64 : cv5 as cls64;  cv4 = Conv(c,32,3) 1x1 -> 4 (direct ltrb distances in grid/stride units)
dist loss (main branch, replaces DFL): CIoU on boxes decoded from clamp(d, min=0) (IoU of an inverted box is undefined)
+ smooth-L1 (beta 1, mean over 4 sides) of RAW d vs the GT ltrb distance (TAL positives have the anchor inside the GT,
so targets >= 0), weighted by the TAL target score like DFL, in the DFL slot (gain 1.5).
The eval forward decodes dist with clamp(min=0) too; the exported model's host decode must do the same."""
import torch, torch.nn as nn, torch.nn.functional as F
import models.yolo as Y
import utils.loss_tal_dual as L
from models.common import Conv
from utils.tal.anchor_generator import make_anchors, dist2bbox, bbox2dist
from utils.torch_utils import de_parallel

SLIM = {'cls64':          dict(box=[(64, 3, 1), (64, 3, 4)], cls=[(64, 3, 1), (64, 3, 1)], boxout='dfl'),
        'distlite_cls64': dict(box=[(32, 3, 1)],             cls=[(64, 3, 1), (64, 3, 1)], boxout='dist')}
_BASE = Y.DualDDetect if not getattr(Y.DualDDetect, '_coral_slim', False) else Y.DualDDetect.__mro__[1]
_BASE_LOSS = L.ComputeLoss if not getattr(L.ComputeLoss, '_coral_slim', False) else L.ComputeLoss.__mro__[1]


def _tower(cin, hidden, cout, gfinal):
    m, c = [], cin
    for o, k, g in hidden: m.append(Conv(c, o, k, g=g)); c = o
    m.append(nn.Conv2d(c, cout, 1, groups=gfinal)); return nn.Sequential(*m)


class DualDDetectSlim(_BASE):
    _coral_slim = True
    ARCH = None   # set by install(); copied to the instance (pickled with it)

    def __init__(self, nc=80, ch=(), inplace=True):
        super().__init__(nc, ch, inplace)
        arch = self.slim_arch = type(self).ARCH; A = SLIM[arch]; self.boxout = A['boxout']
        self.nbox2 = 4 * self.reg_max if self.boxout == 'dfl' else 4
        self.no2 = nc + self.nbox2
        self.cv4 = nn.ModuleList(_tower(x, A['box'], self.nbox2, 4 if self.boxout == 'dfl' else 1) for x in ch[self.nl:])
        self.cv5 = nn.ModuleList(_tower(x, A['cls'], self.nc, 1) for x in ch[self.nl:])

    def forward(self, x):
        shape = x[0].shape
        d1, d2 = [], []
        for i in range(self.nl):
            d1.append(torch.cat((self.cv2[i](x[i]), self.cv3[i](x[i])), 1))
            d2.append(torch.cat((self.cv4[i](x[self.nl + i]), self.cv5[i](x[self.nl + i])), 1))
        if self.training:
            return [d1, d2]
        elif self.dynamic or self.shape != shape:
            self.anchors, self.strides = (t.transpose(0, 1) for t in make_anchors(d1, self.stride, 0.5))
            self.shape = shape
        box, cls = torch.cat([di.view(shape[0], self.no, -1) for di in d1], 2).split((self.reg_max * 4, self.nc), 1)
        dbox = dist2bbox(self.dfl(box), self.anchors.unsqueeze(0), xywh=True, dim=1) * self.strides
        box2, cls2 = torch.cat([di.view(shape[0], self.no2, -1) for di in d2], 2).split((self.nbox2, self.nc), 1)
        dist2 = self.dfl2(box2) if self.boxout == 'dfl' else box2.clamp(min=0)
        dbox2 = dist2bbox(dist2, self.anchors.unsqueeze(0), xywh=True, dim=1) * self.strides
        y = [torch.cat((dbox, cls.sigmoid()), 1), torch.cat((dbox2, cls2.sigmoid()), 1)]
        return y if self.export else (y, [d1, d2])


class DistBboxLoss(nn.Module):
    """BboxLoss for direct ltrb distances: CIoU + smooth-L1 on distances (in the DFL slot)."""
    def forward(self, pred_dist, pred_bboxes, anchor_points, target_bboxes, target_scores, target_scores_sum, fg_mask):
        bbox_mask = fg_mask.unsqueeze(-1).repeat([1, 1, 4])
        pb = torch.masked_select(pred_bboxes, bbox_mask).view(-1, 4)
        tb = torch.masked_select(target_bboxes, bbox_mask).view(-1, 4)
        w = torch.masked_select(target_scores.sum(-1), fg_mask).unsqueeze(-1)
        iou = L.bbox_iou(pb, tb, xywh=False, CIoU=True)
        loss_iou = ((1.0 - iou) * w).sum() / target_scores_sum
        x1y1, x2y2 = torch.split(target_bboxes, 2, -1)
        t_ltrb = torch.cat((anchor_points - x1y1, x2y2 - anchor_points), -1)
        pd = torch.masked_select(pred_dist, bbox_mask).view(-1, 4).float()
        td = torch.masked_select(t_ltrb, bbox_mask).view(-1, 4).float()
        loss_l1 = (F.smooth_l1_loss(pd, td, beta=1.0, reduction='none').mean(-1, keepdim=True) * w).sum() / target_scores_sum
        return loss_iou, loss_l1, iou


class ComputeLossSlim(_BASE_LOSS):
    """ComputeLoss (loss_tal_dual) with a per-branch channel split; main branch 'dist' -> DistBboxLoss."""
    _coral_slim = True

    def __init__(self, model, use_dfl=True):
        super().__init__(model, use_dfl)
        m = de_parallel(model).model[-1]
        self.no2, self.nbox2, self.dist2 = m.no2, m.nbox2, m.boxout == 'dist'
        if self.dist2: self.bbox_loss2 = DistBboxLoss().to(self.device)

    def __call__(self, p, targets, img=None, epoch=0):
        loss = torch.zeros(3, device=self.device)
        feats = p[1][0] if isinstance(p, tuple) else p[0]
        feats2 = p[1][1] if isinstance(p, tuple) else p[1]
        pred_distri, pred_scores = torch.cat([xi.view(feats[0].shape[0], self.no, -1) for xi in feats], 2).split(
            (self.reg_max * 4, self.nc), 1)
        pred_scores = pred_scores.permute(0, 2, 1).contiguous(); pred_distri = pred_distri.permute(0, 2, 1).contiguous()
        pred_distri2, pred_scores2 = torch.cat([xi.view(feats2[0].shape[0], self.no2, -1) for xi in feats2], 2).split(
            (self.nbox2, self.nc), 1)
        pred_scores2 = pred_scores2.permute(0, 2, 1).contiguous(); pred_distri2 = pred_distri2.permute(0, 2, 1).contiguous()
        dtype = pred_scores.dtype
        batch_size = pred_scores.shape[0]
        imgsz = torch.tensor(feats[0].shape[2:], device=self.device, dtype=dtype) * self.stride[0]
        anchor_points, stride_tensor = make_anchors(feats, self.stride, 0.5)
        targets = self.preprocess(targets, batch_size, scale_tensor=imgsz[[1, 0, 1, 0]])
        gt_labels, gt_bboxes = targets.split((1, 4), 2)
        mask_gt = gt_bboxes.sum(2, keepdim=True).gt_(0)
        pred_bboxes = self.bbox_decode(anchor_points, pred_distri)
        pred_bboxes2 = (dist2bbox(pred_distri2.clamp(min=0), anchor_points, xywh=False) if self.dist2
                        else self.bbox_decode(anchor_points, pred_distri2))
        target_labels, target_bboxes, target_scores, fg_mask = self.assigner(
            pred_scores.detach().sigmoid(), (pred_bboxes.detach() * stride_tensor).type(gt_bboxes.dtype),
            anchor_points * stride_tensor, gt_labels, gt_bboxes, mask_gt)
        target_labels2, target_bboxes2, target_scores2, fg_mask2 = self.assigner2(
            pred_scores2.detach().sigmoid(), (pred_bboxes2.detach() * stride_tensor).type(gt_bboxes.dtype),
            anchor_points * stride_tensor, gt_labels, gt_bboxes, mask_gt)
        target_bboxes /= stride_tensor; target_scores_sum = max(target_scores.sum(), 1)
        target_bboxes2 /= stride_tensor; target_scores_sum2 = max(target_scores2.sum(), 1)
        loss[1] = self.BCEcls(pred_scores, target_scores.to(dtype)).sum() / target_scores_sum
        loss[1] *= 0.25
        loss[1] += self.BCEcls(pred_scores2, target_scores2.to(dtype)).sum() / target_scores_sum2
        if fg_mask.sum():
            loss[0], loss[2], _ = self.bbox_loss(pred_distri, pred_bboxes, anchor_points, target_bboxes, target_scores,
                                                 target_scores_sum, fg_mask)
            loss[0] *= 0.25; loss[2] *= 0.25
        if fg_mask2.sum():
            l0, l2, _ = self.bbox_loss2(pred_distri2, pred_bboxes2, anchor_points, target_bboxes2, target_scores2,
                                        target_scores_sum2, fg_mask2)
            loss[0] += l0; loss[2] += l2
        loss[0] *= 7.5; loss[1] *= 0.5; loss[2] *= 1.5
        return loss.sum() * batch_size, loss.detach()


def install(arch):
    if arch not in SLIM: raise ValueError(f'slim arch must be one of {list(SLIM)}')
    DualDDetectSlim.ARCH = arch
    Y.DualDDetect = DualDDetectSlim
    L.ComputeLoss = ComputeLossSlim
    print(f'[coral_slim] DualDDetect -> DualDDetectSlim({arch}), ComputeLoss -> ComputeLossSlim', flush=True)
    return arch


## Step 4: Load the checkpoint


In [ ]:
import torch
if WEIGHTS.startswith('http'):
    local = f'{WORK}/{Path(WEIGHTS).name}'
    if not os.path.exists(local): subprocess.run(['curl', '-fL', '--retry', '5', '-o', local, WEIGHTS], check=True)
    WEIGHTS = local
import coral_slim  # noqa: F401  (needed to unpickle DualDDetectSlim)
ck = torch.load(WEIGHTS, map_location='cpu', weights_only=False)   # trusted file only (full pickle)
model = (ck.get('ema') or ck['model']).float().fuse().eval()
for p in model.parameters(): p.requires_grad_(False)
head = model.model[-1]
assert type(head).__name__ == 'DualDDetectSlim', f'expected a slim-head checkpoint, got {type(head).__name__}'
assert head.boxout == 'dist', f'expected the distlite head (4 box channels), got {head.slim_arch} / {head.boxout}'
acts = {type(m).__name__ for m in model.modules() if type(m).__name__ in ('ReLU6', 'SiLU', 'Hardswish', 'G')}
assert acts == {'ReLU6'}, f'this notebook exports ReLU6 models only, found {acts}'
NC = head.nc
assert NC == len(LABELS), f'model has {NC} classes but LABELS has {len(LABELS)}'
print(f'✓ loaded {WEIGHTS}: head {head.slim_arch}, {NC} classes, activations {acts}, epoch {ck.get("epoch")}')


## Step 5: Export head and ONNX

Only the main branch is exported (`cv4` → 4 ltrb distances, `cv5` → class logits); the training-only aux branch is dropped. Tensors are NHWC so no transposes end up on the TPU.


In [ ]:
import types, onnx, onnxslim, onnxruntime as ort

def head_fn(self, x):
    B, Cl = [], []
    for i in range(self.nl):
        f = x[self.nl + i]                      # main-branch features
        b, c = self.cv4[i](f), self.cv5[i](f)   # (1,4,H,W) ltrb distances, (1,NC,H,W) logits
        if LOGIT_CLAMP > 0: c = torch.clamp(c, -LOGIT_CLAMP, LOGIT_CLAMP)
        B.append(b.permute(0, 2, 3, 1)); Cl.append(c.permute(0, 2, 3, 1))   # NHWC
    if LAYOUT == '6t':
        return tuple(B) + tuple(Cl)
    box = torch.cat([t.flatten(1, 2) for t in B], 1)    # (1, 2100, 4)
    cls = torch.cat([t.flatten(1, 2) for t in Cl], 1)   # (1, 2100, NC)
    return box, cls
head.forward = types.MethodType(head_fn, head)

x0 = torch.zeros(1, 3, IMG, IMG)
with torch.no_grad(): y = model(x0)
print('torch outputs:', [tuple(t.shape) for t in y])
ONNX_PATH = OUT / f'{OUT_NAME}.onnx'
torch.onnx.export(model, x0, str(ONNX_PATH), opset_version=14, input_names=['images'],
                  output_names=[f'out{i}' for i in range(len(y))], dynamo=False)
mo = onnxslim.slim(onnx.load(str(ONNX_PATH))); onnx.checker.check_model(mo); onnx.save(mo, str(ONNX_PATH))

xr = np.random.RandomState(0).rand(1, 3, IMG, IMG).astype(np.float32)
with torch.no_grad(): yt = [t.numpy() for t in model(torch.from_numpy(xr))]
yo = ort.InferenceSession(str(ONNX_PATH)).run(None, {'images': xr})
diff = max(float(np.abs(p - q).max()) for p, q in zip(yt, yo))
print(f'✓ ONNX written: {ONNX_PATH}  (max |onnx - torch| = {diff:.2e})')
assert diff < 1e-3


## Step 6: Calibration data

Full-int8 quantization picks one scale per tensor from these frames. Frames that look like your cameras (including night and IR) give the best int8 accuracy.


In [ ]:
import cv2
if CALIB_NPY:
    calib = np.load(CALIB_NPY).astype(np.float32)
else:
    if not CALIB_IMAGE_DIR:
        if not os.path.isdir(f'{WORK}/coco128'):
            subprocess.run(f'cd {WORK} && curl -fsSL -o coco128.zip https://ultralytics.com/assets/coco128.zip && unzip -q coco128.zip', shell=True, check=True)
        CALIB_IMAGE_DIR = f'{WORK}/coco128/images'
    files = sorted(f for f in glob.glob(f'{CALIB_IMAGE_DIR}/**/*', recursive=True) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    assert files, f'no images under {CALIB_IMAGE_DIR}'
    rng = np.random.RandomState(0)
    pick = [files[i] for i in rng.permutation(len(files))[:CALIB_N]]
    calib = np.zeros((len(pick), IMG, IMG, 3), np.float32)
    for k, f in enumerate(pick):
        im = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)
        im = cv2.resize(im, (IMG, IMG), interpolation=cv2.INTER_LINEAR)   # stretch, like Frigate's region resize
        if rng.rand() < CALIB_GRAY_SHARE: im = np.repeat(cv2.cvtColor(im, cv2.COLOR_RGB2GRAY)[..., None], 3, 2)
        calib[k] = im
    CALIB_NPY = str(OUT / f'calib{len(calib)}_{IMG}.npy'); np.save(CALIB_NPY, calib)
assert calib.ndim == 4 and calib.shape[1:] == (IMG, IMG, 3) and calib.max() > 1.5, calib.shape
print(f'✓ calibration: {calib.shape} from {CALIB_NPY}')


## Step 7: ONNX → SavedModel → full-int8 TFLite

Input is uint8 RGB 0..255 (Frigate `input_dtype: int`), outputs are int8. Each output tensor gets its own quantization scale, so the box distances and the class logits keep their own ranges.


In [ ]:
import shutil, onnx2tf, tensorflow as tf
SM = OUT / f'{OUT_NAME}_savedmodel'
shutil.rmtree(SM, ignore_errors=True)
os.chdir(OUT)
# onnx2tf looks for this sample file even when it does not quantize
np.save('calibration_image_sample_data_20x128x128x3_float32.npy', np.zeros((20, 128, 128, 3), np.float32))
onnx2tf.convert(
    input_onnx_file_path=str(ONNX_PATH), output_folder_path=str(SM), output_integer_quantized_tflite=False,
    custom_input_op_name_np_data_path=[['images', CALIB_NPY, np.array([[[[0, 0, 0]]]], np.float32),
                                        np.array([[[[255, 255, 255]]]], np.float32)]],
    not_use_onnxsim=True, verbosity='error', output_signaturedefs=True, disable_group_convolution=True,
    batch_size=1, copy_onnx_input_output_names_to_tflite=True)

def representative_dataset():
    for i in range(len(calib)):
        yield [(calib[i:i + 1] / 255.0).astype(np.float32)]
conv = tf.lite.TFLiteConverter.from_saved_model(str(SM))
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = representative_dataset
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.uint8
conv.inference_output_type = tf.int8
INT8 = OUT / f'{OUT_NAME}.tflite'
INT8.write_bytes(conv.convert())
shutil.rmtree(SM, ignore_errors=True); os.chdir(WORK)

it = tf.lite.Interpreter(str(INT8)); it.allocate_tensors()
print('input :', it.get_input_details()[0]['shape'], it.get_input_details()[0]['dtype'])
for d in it.get_output_details():
    print('output:', d['shape'], d['dtype'], 'scale %.4f zp %d' % d['quantization'])
print(f'✓ int8 TFLite: {INT8} ({INT8.stat().st_size / 1e6:.2f} MB)')


## Step 8: Check the int8 model on a test image (CPU)

This decodes the outputs the same way the Frigate plugin does: filter on the max class logit, decode boxes with `max(d, 0)` and the anchor grid, NMS, then sigmoid on the kept boxes.


In [ ]:
from PIL import Image, ImageDraw
TEST_IMAGE = f'{WORK}/test.jpg'
if not os.path.exists(TEST_IMAGE):
    subprocess.run(['curl', '-fsSL', '-o', TEST_IMAGE, 'https://ultralytics.com/images/bus.jpg'], check=False)
MIN_SCORE, NMS_IOU = 0.4, 0.4

def anchors_strides(img):
    a, s = [], []
    for st in (8, 16, 32):
        h = w = img // st
        gy, gx = np.meshgrid(np.arange(h, dtype=np.float32), np.arange(w, dtype=np.float32), indexing='ij')
        a.append(np.stack((gx.ravel(), gy.ravel()), 1) + 0.5); s.append(np.full((h * w, 1), st, np.float32))
    return np.concatenate(a), np.concatenate(s)
ANCH, STRD = anchors_strides(IMG)

def detect(interp, rgb):
    x = cv2.resize(rgb, (IMG, IMG), interpolation=cv2.INTER_LINEAR)[None].astype(np.uint8)
    interp.set_tensor(interp.get_input_details()[0]['index'], x); interp.invoke()
    outs = interp.get_output_details()
    if LAYOUT == '6t':   # flatten per-scale NHWC tensors to (2100, C), scale order 8, 16, 32
        deq = lambda d: (interp.get_tensor(d['index'])[0].astype(np.float32) - d['quantization'][1]) * d['quantization'][0]
        bx = sorted([d for d in outs if d['shape'][-1] == 4], key=lambda d: -d['shape'][1])
        cl = sorted([d for d in outs if d['shape'][-1] != 4], key=lambda d: -d['shape'][1])
        box = np.concatenate([deq(d).reshape(-1, 4) for d in bx]); cls = np.concatenate([deq(d).reshape(-1, NC) for d in cl])
    else:
        bd = next(d for d in outs if d['shape'][-1] == 4); cd = next(d for d in outs if d['shape'][-1] == NC)
        box = (interp.get_tensor(bd['index'])[0].astype(np.float32) - bd['quantization'][1]) * bd['quantization'][0]
        cls = (interp.get_tensor(cd['index'])[0].astype(np.float32) - cd['quantization'][1]) * cd['quantization'][0]
    mx = cls.max(1); m = mx >= np.log(MIN_SCORE / (1 - MIN_SCORE))
    if not m.any(): return []
    d = np.maximum(box[m], 0)                     # distlite decode
    a, s = ANCH[m], STRD[m]
    xyxy = np.concatenate(((a - d[:, :2]) * s, (a + d[:, 2:]) * s), 1)
    xywh = np.concatenate((xyxy[:, :2], xyxy[:, 2:] - xyxy[:, :2]), 1)
    keep = np.array(cv2.dnn.NMSBoxes(xywh.tolist(), (mx[m] + 30).tolist(), 0.0, NMS_IOU), np.int32).ravel()
    return [(LABELS[int(cls[m][k].argmax())], float(1 / (1 + np.exp(-mx[m][k]))), xyxy[k] / IMG) for k in keep[:20]]

if os.path.exists(TEST_IMAGE) and os.path.getsize(TEST_IMAGE) > 1000:
    rgb = cv2.cvtColor(cv2.imread(TEST_IMAGE), cv2.COLOR_BGR2RGB)
    dets = detect(it, rgb)
    for lab, sc, b in dets: print(f'{lab:10s} {sc:.2f} {np.round(b, 3)}')
    im = Image.fromarray(rgb); dr = ImageDraw.Draw(im); W_, H_ = im.size
    for lab, sc, b in dets:
        dr.rectangle([b[0] * W_, b[1] * H_, b[2] * W_, b[3] * H_], outline=(255, 0, 0), width=3)
        dr.text((b[0] * W_ + 3, b[1] * H_ + 3), f'{lab} {sc:.2f}', fill=(255, 0, 0))
    im.save(OUT / 'test_detections.jpg')
    try:
        from IPython.display import display; display(im.resize((640, int(640 * H_ / W_))))
    except Exception: pass
else:
    print('no test image; skipped')


## Step 9: Compile for the EdgeTPU


In [ ]:
# [colab-only]
!curl -fsSL https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo gpg --dearmor -o /usr/share/keyrings/coral-edgetpu.gpg
!echo "deb [signed-by=/usr/share/keyrings/coral-edgetpu.gpg] https://packages.cloud.google.com/apt coral-edgetpu-stable main" | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list
!sudo apt-get update -q && sudo apt-get install -q -y edgetpu-compiler


In [ ]:
EDGETPU_COMPILER = 'edgetpu_compiler'   # tested with Edge TPU Compiler 16.0
r = subprocess.run([EDGETPU_COMPILER, '--show_operations', '--search_delegate', '--delegate_search_step', '30',
                    '--timeout_sec', '180', '-o', str(OUT), str(INT8)], capture_output=True, text=True)
print(r.stdout[-4000:])
assert r.returncode == 0, r.stderr
EDGETPU = OUT / f'{OUT_NAME}_edgetpu.tflite'
assert EDGETPU.exists()
assert 'Number of Edge TPU subgraphs: 1' in r.stdout, 'expected one EdgeTPU subgraph'
not_mapped = [l for l in r.stdout.splitlines() if l.strip() and ('not' in l.lower() and 'mapped' in l.lower())]
print('ops not on the TPU:', not_mapped or 'none')
print(f'✓ EdgeTPU model: {EDGETPU} ({EDGETPU.stat().st_size / 1e6:.2f} MB)')


## Step 10: Download and use in Frigate

```yaml
detectors:
  coral:
    type: edgetpu
    device: usb          # or pci

model:
  model_type: yolo-generic
  width: 320
  height: 320
  input_tensor: nhwc
  input_dtype: int
  path: /config/model_cache/yolov9-s-relu6-slim_320_int8_edgetpu.tflite
  labelmap_path: /config/model_cache/labels-coco17.txt
```

Mount the patched `edgetpu_tfl.py` from this repository as described in its README. At start-up the Frigate log
should show `YOLO box decode: distlite (boxes tensor (1, 2100, 4), class tensor (1, 2100, 17))`.


In [ ]:
# [colab-only]
(OUT / 'labels-coco17.txt').write_text('\n'.join(LABELS) + '\n')
from google.colab import files
for f in (EDGETPU, INT8, OUT / 'labels-coco17.txt'):
    files.download(str(f))
